In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from matplotlib import cm
c_map = cm.jet

import sys
sys.path.append('../experiments/')


import ssp_bayes_opt
import figure_utils as utils

In [53]:
def objective(x):
 return (1.4 - 3.0 * x + 2. * x**2) * np.sin(18.0 * x)

target = objective
pbounds = np.array([[0, 1.2 ]])

plt.figure(figsize=(2,1))
plt.plot(np.linspace(pbounds[0,0],pbounds[0,1]), objective(np.linspace(pbounds[0,0],pbounds[0,1])))


In [57]:



ls = 0.1
beta_ucb=1.
gamma_c=0
optimizer = ssp_bayes_opt.BayesianOptimization(f=target,
                                               bounds=pbounds, 
                                               verbose=True,
                                               sampling_seed=10)
optimizer.maximize(init_points=1, 
                   n_iter=9,
                   num_restarts=1,
                   agent_type='ssp-hex',
                   ssp_dim=75,decoder_method='direct-optim', length_scale=ls,beta_ucb=beta_ucb,gamma_c=gamma_c,seed=10)

vals = np.zeros((len(optimizer.res),))
sample_locs = []

for i, res in enumerate(optimizer.res):
    vals[i] = res['target']
    sample_locs.append(res['params'])
    
samples1=[np.array(sample_locs).reshape(-1), np.array(vals)]
xs = np.linspace(pbounds[0,0],pbounds[0,1], 100).reshape(-1,1)
phis = optimizer.agt.encode(xs)
mu1, var1 = optimizer.agt.blr.predict(phis)


In [58]:
optimizer = ssp_bayes_opt.BayesianOptimization(f=target,
                                               bounds=pbounds, 
                                               verbose=True,
                                               sampling_seed=10)
optimizer.maximize(init_points=1, 
                   n_iter=19,
                   num_restarts=1,
                   agent_type='ssp-hex',
                   ssp_dim=75,decoder_method='direct-optim', length_scale=ls,beta_ucb=beta_ucb,gamma_c=gamma_c,seed=10)

vals = np.zeros((len(optimizer.res),))
sample_locs = []

for i, res in enumerate(optimizer.res):
    vals[i] = res['target']
    sample_locs.append(res['params'])
    
samples2=[np.array(sample_locs).reshape(-1), np.array(vals)]
mu2, var2 = optimizer.agt.blr.predict(phis)



In [73]:



# fig, axs = plt.subplots(1,3, figsize=(utils.doublecolwidth,2))

fig = plt.figure(layout="constrained", figsize=(utils.doublecolwidth,1.7))

gs = fig.add_gridspec(1,4, width_ratios=[1,1,1,0.5])
axs = gs.subplots(sharex=True, sharey=True)

for i in range(3):
    axs[i].set_ylim([-3,2.5])
    axs[i].set_xlim([pbounds[0,0],pbounds[0,1]])
    axs[i].set_xlabel('$x$')
axs[-1].set_axis_off()

sample_id = 3

axs[0].plot(samples1[0][sample_id], samples1[1][sample_id], "o", color=utils.reds[1], label='Sample points', markersize=3,zorder=5)

axs[0].plot(xs,target(xs), "-", color=utils.grays[2], label='$f(x)$')

kern = phis  @ optimizer.agt.encode(np.array(samples1[0][sample_id]).reshape(1,-1)).T

axs[0].plot(xs.reshape(-1), samples1[1][sample_id]*kern.reshape(-1), '--',color=utils.cols[1], label='$f(x_0)[\phi(x_0) \cdot \phi(x)]$')
# axs[0].legend()


mu, var = optimizer.agt.blr.predict(phis)
axs[1].plot(xs,target(xs), "-", color=utils.grays[2])
axs[1].plot(xs.reshape(-1), mu1.reshape(-1), '--', color=utils.cols[0], label='BLR mean, $\mu$')
axs[1].fill_between(xs.reshape(-1),  mu1.reshape(-1)-np.sqrt(var1), 
                 mu1.reshape(-1)+np.sqrt(var1), alpha=0.2, edgecolor=None, facecolor=utils.cols[0], label='BLR SD, $\sigma$')

axs[1].plot(xs.reshape(-1),  mu1.reshape(-1)+ beta_ucb*np.sqrt(var1), ':,',color=utils.cols[2], label='UCB')
   
axs[1].plot(samples1[0], samples1[1], "o", color=utils.reds[1], markersize=3)
# axs[1].legend( fontsize=9)


axs[2].plot(xs,target(xs), "-", color=utils.grays[2])
axs[2].plot(xs.reshape(-1), mu2.reshape(-1), '--', color=utils.cols[0])
axs[2].fill_between(xs.reshape(-1),  mu2.reshape(-1)-np.sqrt(var2), 
                 mu2.reshape(-1)+np.sqrt(var2), alpha=0.2, edgecolor=None, facecolor=utils.cols[0])
   
axs[2].plot(samples2[0], samples2[1], "o", color=utils.reds[1], markersize=3)
axs[2].plot(xs.reshape(-1),  mu2.reshape(-1)+ beta_ucb*np.sqrt(var2), ':', color=utils.cols[2], label='UCB')


handles0, labels0 = axs[0].get_legend_handles_labels()
handles1, labels1 = axs[1].get_legend_handles_labels()

axs[-1].legend(handles0+handles1, labels0+labels1, loc='center left',fontsize=9)

axs[0].set_title("\\textbf{a} $\quad$ Kernel at a single sample",fontsize=9)
axs[1].set_title("\\textbf{b} $\quad$ BLR after 10 samples",fontsize=9)
axs[2].set_title( "\\textbf{c} $\quad$ BLR after 20 samples",fontsize=9)

utils.save(fig,"BLR_example.pdf")

In [74]:
def make_laplace_freqs(dim, eps=1e-3, rng=np.random):
    a = rng.standard_cauchy(size=(dim - 1) // 2)
    #a = rng.exponential(scale=(dim-1) // 2)
    sign = rng.choice((-1, +1), len(a))
    phi = sign * np.pi * (eps + a * (1 - 2 * eps))

    fv = np.zeros(dim, dtype='complex64')
    
    fv[0] = 1
    fv[1:(dim + 1) // 2] = phi #np.cos(phi) + 1j * np.sin(phi)
    fv[-1:dim // 2:-1] = -phi #np.conj(fv[1:(dim + 1) // 2])
    if dim % 2 == 0:
        fv[dim // 2] = 1
    return fv, phi


laplace_phase_mat, a = make_laplace_freqs(1000)


In [75]:
laplace_enc = ssp_bayes_opt.sspspace.SSPSpace(1,len(laplace_phase_mat), 
                                              phase_matrix=laplace_phase_mat[:,None], length_scale=2.)

origin = laplace_enc.encode(np.array([[0]]))
xs = np.linspace(-4,4,1000)

phis = laplace_enc.encode(xs[:,None])

sims = np.sum(phis * origin, axis=-1)

# plt.subplot(1,3,1)
# plt.hist(a, bins=100, density=True)
# plt.title('Frequency histograms')
# plt.subplot(1,3,2)
plt.figure(figsize=(3,2))
plt.plot(xs, sims)
plt.title('Similarity')

In [76]:
def make_sin_freqs(dim, eps=1e-3, rng=np.random):
#     a = rng.standard_cauchy(size=(dim - 1) // 2)
    a = np.ones(((dim-1) // 2,)) #* np.pi
    sign = rng.choice((-1, +1), len(a))
    phi = sign * np.pi * (eps + a * (1 - 2 * eps))

    fv = np.zeros(dim, dtype='complex64')
    
    fv[0] = 0
    fv[1:(dim + 1) // 2] = phi #np.cos(phi) + 1j * np.sin(phi)
    fv[-1:dim // 2:-1] = -phi #np.conj(fv[1:(dim + 1) // 2])
    if dim % 2 == 0:
        fv[dim // 2] = 0
    return fv, phi

sin_phase_mat, a = make_sin_freqs(1000)
sin_enc = ssp_bayes_opt.sspspace.SSPSpace(1,len(sin_phase_mat), 
                                              phase_matrix=sin_phase_mat[:,None], length_scale=1.)


origin = sin_enc.encode(np.array([[0]]))
xs = np.linspace(-4,4,1000)

phis = sin_enc.encode(xs[:,None])

sims = np.sum(phis * origin, axis=-1)

plt.figure(figsize=(6,1))
plt.subplot(1,3,1)
plt.hist(a, bins=100, density=True)
plt.title('Frequency histograms')
plt.subplot(1,3,2)
plt.plot(xs, sims)
plt.title('Similarity')
plt.subplot(1,3,3)
ffts = np.fft.fft(phis, axis=1)
mean_mags = np.mean(np.abs(ffts), axis=0)
std_mags = np.std(np.abs(ffts), axis=0)
plt.plot(mean_mags)
plt.fill_between(np.arange(1000), y1=mean_mags - std_mags, y2=mean_mags + std_mags, alpha = 0.2)
plt.title('Fourier coefficient\n magnitude')

In [79]:
laplace_origin = laplace_enc.encode(np.array([[0]]))
sin_origin = sin_enc.encode(np.array([[0]]))
origin = laplace_origin * sin_origin


xs = np.linspace(-4,4,200)

laplace_phis = laplace_enc.encode(xs[:,None])
sin_phis = sin_enc.encode(xs[:,None])

phis = laplace_phis * sin_phis

sims1 = np.sum(phis * origin, axis=-1)
laplace_sims1 = np.sum(laplace_phis * laplace_origin, axis=-1)
sin_sims1 = np.sum(sin_phis * sin_origin, axis=-1)

sum_phis1 = (laplace_phis + sin_phis)
sum_origin1 = (laplace_origin + sin_origin)
sum_sims1 = np.sum(sum_phis1 * sum_origin1, axis=-1) 



length_scale = 4

x1_space = ssp_bayes_opt.sspspace.RandomSSPSpace(domain_dim=1,ssp_dim=1000,length_scale=length_scale)
x2_space = ssp_bayes_opt.sspspace.RandomSSPSpace(domain_dim=1,ssp_dim=1000,length_scale=length_scale)

x1s = np.linspace(-20,20,100)
x2s = np.linspace(-20,20,100)
X1, X2 = np.meshgrid(x1s, x2s)
query_xs = np.vstack((X1.flatten(), X2.flatten())).T


def prod_encode(x1,x2,enc1,enc2):
    phi1 = enc1.encode(x1)
    phi2 = enc2.encode(x2)
    return x1_space.bind(phi1 , phi2)

def sum_encode(x1,x2,enc1,enc2):
    phi1 = enc1.encode(x1)
    phi2 = enc2.encode(x2)
    return (phi1 + phi2) / 2

# x1_origin = x1_space.encode([[10]])
# x2_origin = x2_space.encode([[10]])

prod_origin = prod_encode(np.array([[0]]),np.array([[0]]),x1_space, x2_space)
sum_origin = sum_encode(np.array([[0]]),np.array([[0]]),x1_space, x2_space)


query_sum_phis = sum_encode(query_xs[:,0][:,None], query_xs[:,1][:,None], x1_space, x2_space)
query_prod_phis = prod_encode(query_xs[:,0][:,None], query_xs[:,1][:,None], x1_space, x2_space)

neg_sum_origin = sum_encode(np.array([[-10]]),np.array([[-10]]), x1_space, x2_space)
neg_prod_origin = prod_encode(np.array([[-10]]),np.array([[-10]]), x1_space, x2_space)

pos_sum_origin = sum_encode(np.array([[10]]),np.array([[10]]), x1_space, x2_space)
pos_prod_origin = prod_encode(np.array([[10]]),np.array([[10]]), x1_space, x2_space)

neg_sum_sims = np.sum(query_sum_phis * 
                      x1_space.bind(sum_origin , neg_sum_origin) ,axis=-1)
neg_prod_sims = np.sum(query_prod_phis * x1_space.bind(prod_origin , neg_prod_origin),axis=-1)

sum_sims = np.sum(query_sum_phis * sum_origin, axis=-1)
prod_sims = np.sum(query_prod_phis * prod_origin ,axis=-1)

pos_sum_sims = np.sum(query_sum_phis * x1_space.bind(sum_origin , pos_sum_origin) ,axis=-1)
pos_prod_sims = np.sum(query_prod_phis * x1_space.bind(prod_origin , pos_prod_origin) ,axis=-1)



In [87]:

c_map='turbo'
compositional_fig = plt.figure(figsize=(utils.doublecolwidth, 3.3),dpi=300)
subfigs = compositional_fig.subfigures(2, 1, wspace=0.07, height_ratios=[1, 3], hspace=0.1)
font_size = 9

axs_top = subfigs[0].subplots(1, 4)
# plt.figure(figsize=(10,3))
# plt.subplot(1,4,1)
axs_top[0].plot(xs, sin_sims1,color=utils.cols[0])
axs_top[0].spines['top'].set_visible(False)
axs_top[0].spines['right'].set_visible(False)
axs_top[0].set_xticklabels([])
axs_top[0].set_yticks([])
axs_top[0].set_title('PER', fontsize=font_size)

# plt.subplot(1,4,2)
axs_top[1].plot(xs, laplace_sims1, lw=1,color=utils.cols[0])
axs_top[1].spines['top'].set_visible(False)
axs_top[1].spines['right'].set_visible(False)
axs_top[1].set_xticklabels([])
axs_top[1].set_yticks([])
axs_top[1].set_title('LaPlace', fontsize=font_size)

# plt.subplot(1,4,3)
axs_top[2].plot(xs, sims1, lw=1,color=utils.cols[0])
axs_top[2].spines['top'].set_visible(False)
axs_top[2].spines['right'].set_visible(False)
axs_top[2].set_title(r'PER $\circledast$ LaPlace', fontsize=font_size)
axs_top[2].set_xticklabels([])
axs_top[2].set_yticks([])


# plt.subplot(1,4,4)
axs_top[3].plot(xs, sum_sims1, lw=1,color=utils.cols[0])
axs_top[3].spines['top'].set_visible(False)
axs_top[3].spines['right'].set_visible(False)
axs_top[3].set_title(r'PER + LaPlace', fontsize=font_size)
axs_top[3].set_xticklabels([])
axs_top[3].set_yticks([])



_axs_bot = subfigs[1].subplots(2, 5, subplot_kw={"projection": "3d"}, 
                               width_ratios=[0.3,1,1,1,0.3],
                              gridspec_kw=dict(top=1, left=0, right=1, bottom=0,hspace=-0.3,wspace=-0.2))

axs_bot = _axs_bot[:,1:-1]
for ax in _axs_bot[:,0]:
    ax.set_axis_off()
for ax in _axs_bot[:,-1]:
    ax.set_axis_off()

axs_bot[0,0].plot_surface(X1, X2, neg_sum_sims.reshape((100,100)), cmap=c_map, clip_on=False)
axs_bot[0,0].set_title(r'$\phi(\mathbf{x}_{0}) \circledast \phi(-\Delta\mathbf{x})$', fontsize=font_size,y=0.9)

axs_bot[0,1].plot_surface(X1, X2, sum_sims.reshape((100,100)), cmap=c_map, clip_on=False)
axs_bot[0,1].set_title(r'$\phi(\mathbf{x}_{0})$', fontsize=font_size,y=0.9)

axs_bot[0,2].plot_surface(X1, X2, pos_sum_sims.reshape((100,100)), cmap=c_map, clip_on=False)
axs_bot[0,2].set_title(r'$\phi(\mathbf{x}_{0}) \circledast \phi(\Delta\mathbf{x})$', fontsize=font_size,y=0.9)



axs_bot[1,0].plot_surface(X1, X2, neg_prod_sims.reshape((100,100)), cmap=c_map, clip_on=False)
# ax[1,0].set_title(r'$\mathrm{sinc}(x_{1}) \circledast \mathrm{sinc}(x_{2})$')

axs_bot[1,1].plot_surface(X1, X2, prod_sims.reshape((100,100)), cmap=c_map, clip_on=False)
# ax[1,1].set_title(r'$\mathrm{sinc}(x_{1}) \circledast \mathrm{sinc}(x_{2})$')

axs_bot[1,2].plot_surface(X1, X2, pos_prod_sims.reshape((100,100)), cmap=c_map, clip_on=False)
# ax[1,2].set_title()

z_label = [r'$\mathrm{sinc}(x_{1}) + \mathrm{sinc}(x_{2})$',r'$\mathrm{sinc}(x_{1}) \circledast \mathrm{sinc}(x_{2})$']

for row in range(axs_bot.shape[0]):
    for col in range(axs_bot.shape[1]):
        axs_bot[row,col].set(xticklabels=[],
                   yticklabels=[],
                   zticklabels=[])
        axs_bot[row,col].set_xlabel(r'$x_{1}$', fontsize=7,clip_on=False)
        axs_bot[row,col].set_ylabel(r'$x_{2}$', fontsize=7,clip_on=False)
        axs_bot[row,col].xaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
        axs_bot[row,col].yaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
        axs_bot[row,col].zaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
        axs_bot[row,col].set_box_aspect(aspect=(1.,1.,0.5), zoom=1.1)
        axs_bot[row,col].set_proj_type('ortho')

        axs_bot[row,col].set_xlim((np.min(X1),np.max(X1)))
        axs_bot[row,col].set_ylim((np.min(X2),np.max(X2)))
        axs_bot[row,col].patch.set_alpha(0)
        
        
        tmp_planes = axs_bot[row,col].zaxis._PLANES 
        axs_bot[row,col].zaxis._PLANES = ( tmp_planes[2], tmp_planes[3], 
                     tmp_planes[0], tmp_planes[1], 
                     tmp_planes[4], tmp_planes[5])
        # rotate label
        axs_bot[row,col].zaxis.set_rotate_label(False)  # disable automatic rotation
        
        if col == 0:
            axs_bot[row,col].set_zlabel(z_label[row], rotation=90, fontsize=7)
        axs_bot[row,col].grid(False)
        axs_bot[row,col].xaxis.labelpad = -15
        axs_bot[row,col].yaxis.labelpad = -15
        axs_bot[row,col].zaxis.labelpad = -15
    ### end for
### end for
# kernels_2d = plt.gcf()
# figure_utils.save(kernels_2d, 'shifting-encoding.pdf')
subfigs[0].text(0., 0.93, '\\textbf{a}', size=9, va="baseline", ha="left")
subfigs[1].text(0., 0.93, '\\textbf{b}', size=9, va="baseline", ha="left")


compositional_fig.tight_layout()
utils.save(compositional_fig,"comp_kernel_example.pdf")
# compositional_fig.savefig("comp_kernel_example.pdf")

In [103]:
axs_bot[row,col].